# 03 ML Pipeline: Customer Tribe Discovery

This notebook starts from the prepared parquet outputs created by Notebook 02. It does not repeat raw loading, cleaning, or exploratory data quality work from Notebooks 01 and 02.

The goal is to discover product-first customer tribes from purchase behavior. The client hypothesis is roughly 10-15 tribes, but that range is not used as a modeling constraint. The final recommendation is selected from metrics, stability-ready diagnostics, cluster balance, product/sector lift interpretability, and business usefulness.

## Table of Contents

| Section | What it covers |
|---|---|
| [Experiment Workbench](#experiment-workbench) | Points optional tuning work to the sandbox notebook so this notebook remains the official pipeline. |
| [Stage 0: Load Prepared Data](#stage-0) | Sets the run mode, validates paths, loads prepared transactions, and previews the input data. |
| [Stage 0.5: Expensive Cache Audit](#stage-0-5) | Checks whether Stage 1-6 artifacts can be reused safely from cache. |
| [Stage 1: Basket Construction](#stage-1) | Converts checkout records into basket sentences for product embedding training. |
| [Stage 2: Item2Vec Product Embeddings](#stage-2) | Builds or loads product embeddings from product co-purchase behavior. |
| [Stage 3: Product Embedding Validation](#stage-3) | Reviews product-neighbor diagnostics and embedding quality checks. |
| [Stage 4: Customer Embeddings](#stage-4) | Creates product-first customer embeddings from purchased products and quantities. |
| [Stage 5: Behavioral Features](#stage-5) | Builds model-ready feature sets and supporting non-demographic behavior summaries. |
| [Stage 6: Candidate Model Comparison](#stage-6) | Compares candidate clustering solutions and exports model diagnostics. |
| [Stage 7: Cluster Validity and Stability](#stage-7) | Tests whether the selected solution remains defensible under validity and stability checks. |
| [Stage 8: Tribe Profiling and Interpretability](#stage-8) | Explains tribes using product, sector, theme, and term lift evidence. |
| [Stage 9: Final Model Selection and Shopping Mission Summary](#stage-9) | Selects the final tribe solution and summarizes actionable shopping missions. |
| [Decision Log: Final Tribe Model Selection](#decision-log) | Records the final model choice and rationale for handoff. |
| [Stage 10: Additional Business Lens](#stage-10) | Translates the selected tribes into campaign opportunities, KPI tests, and conservative financial sizing. |

Tribe discovery is documented as a clear late-stage progression:

| Layer | Pipeline stage | What it proves | Main outputs |
|---|---|---|---|
| 1. Organic core tribes | Stage 6, checked in Stage 7 | Dense product-purchase behavior groups exist without forcing every customer into a tribe. | Candidate assignments, model diagnostics, stability report |
| 2. Confidence-scored soft assignment | Stage 6 export logic, reviewed in Stage 8/9 | Nearby non-core customers can be attached for campaign usability while preserving assignment provenance and confidence. | Final assignments, assignment provenance, confidence fields |
| 3. Evidence-backed profiling | Stage 8/9 | Each tribe can be explained through lifted products, sectors, themes, and discovered product-term signals. | Tribe profiles, compact core tribe table |
| 4. Selected shopping missions | Stage 9 | A capped set of large, strongly backed activation audiences emerges from product evidence across the core tribes. | Shopping mission table, mission overview chart, core-tribe-by-mission lift heatmap |
| 5. Business opportunity lens | Stage 10 | Evidence-backed audiences can be turned into CRM, campaign, retention, and promo-efficiency tests. | Campaign opportunity matrix, financial sizing template, business lens report |

Read the official result in that order: first whether the core tribes are real, then how much soft assignment was needed for coverage, then whether the profile evidence supports the business story, which selected shopping missions are actionable for campaigns, and finally how those audiences translate into business tests and financial opportunity sizing.

<a id="experiment-workbench"></a>

## Experiment Workbench

Optional hyperparameter experiments now live in `04_experiment_sandbox.ipynb`. Use that notebook to test variants, promote the winning settings into YAML, then rerun this official pipeline notebook to produce clean artifacts.


<a id="stage-0"></a>

## Stage 0: Load Prepared Data

Notebook 03 consumes `df_combined.parquet`, validates the fields required for ML, and merges product metadata only if the prepared file does not already contain it.

In [ ]:
import os

# Mode toggle: this cell is authoritative, even if src.config was imported earlier in the kernel.
RUN_MODE = "dev"  # change to "prod" for the full production pipeline
os.environ["CARREFOUR_MODE"] = RUN_MODE.strip().lower()

from IPython.display import Image, Markdown, display

from pathlib import Path
import sys

for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "src").is_dir():
        project_root = candidate
        break
else:
    project_root = Path.cwd().resolve()

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import configure_mode
from src.cache_audit import assert_mode_path_audit
from src.data_loader import load_prepared_transactions, peek
from src.stage_reports import display_stage_report, write_stage_report
from src.utils import set_global_seed

previous_mode = globals().get("MODE")
CONFIG = configure_mode(RUN_MODE)
MODE = CONFIG.mode
DATA_PROCESSED = CONFIG.data_processed
MODELS = CONFIG.models
OUTPUTS = CONFIG.outputs

if previous_mode and previous_mode != MODE:
    for stale_name in [
        "transactions",
        "basket_path",
        "item2vec_model",
        "product_embeddings_path",
        "embedding_validation_csv",
        "embedding_validation_detail_md",
        "customer_embeddings_path",
        "behavior_path",
        "feature_sets",
        "model_suite",
        "stage6_diagnostics",
        "cluster_validity",
        "validity_table",
        "stage6_winner_key",
        "stage7_projection_dir",
        "stage7_projection_figures",
        "profile_paths",
        "selected_key",
        "comparison_path",
        "selected",
        "selected_assignment_path",
        "selected_profile_path",
        "stage8_figures_dir",
        "tribe_vs_population_dashboard",
        "theme_lift_heatmap",
        "presentation_dir",
        "evidence_dir",
        "presentation_figures_dir",
        "assignment_export",
        "profile_export",
        "decision_log_path",
        "cluster_summary_paths",
        "mission_microtribe_paths",
        "figures",
    ]:
        globals().pop(stale_name, None)

set_global_seed(CONFIG.random_seed)
CONFIG.ensure_directories()
mode_path_audit = assert_mode_path_audit(CONFIG)

transactions = load_prepared_transactions(cfg=CONFIG)
print(f"Run mode: {MODE}")
print(f"Data path: {DATA_PROCESSED}")
print(f"Model path: {MODELS}")
print(f"Output path: {OUTPUTS}")
peek(transactions, 3)

from src.visualization import plot_prepared_data_overview

stage0_figure = plot_prepared_data_overview(transactions, cfg=CONFIG)
stage0_report = write_stage_report(
    "00",
    "Load Prepared Data",
    summary=[
        f"Run mode active: {MODE}",
        "Mode/path audit passed; dev and prod namespaces are not mixed.",
        "Prepared transactions loaded from the configured mode path.",
    ],
    metrics={
        "mode_path_checks": mode_path_audit.height,
        "failed_mode_path_checks": mode_path_audit.filter(mode_path_audit.get_column("status") == "fail").height,
    },
    figures={"Prepared data overview": stage0_figure},
    artifacts={"Prepared transactions": CONFIG.prepared_transactions_path},
    cfg=CONFIG,
)
display_stage_report(stage0_report)
display(mode_path_audit.select(["check", "status", "exists", "reason"]).head(12))
display(Image(filename=str(stage0_figure)))


<a id="stage-0-5"></a>

## Stage 0.5: Expensive Cache Audit

Before running the expensive pipeline stages, this table checks whether Stage 1-6 artifacts will be reused from cache. A cache hit requires the artifact to exist and its metadata hash to match the current input files and config.

In [ ]:
import importlib
import polars as pl

import src.utils
importlib.reload(src.utils)
import src.cache_audit
importlib.reload(src.cache_audit)
from src.cache_audit import stage_1_6_cache_audit
from src.stage_reports import display_stage_report, write_stage_report
# Optional: if existing artifacts were produced from the current data/config but metadata is missing,
# uncomment the next two lines once to adopt them into the cache manifest.
# from src.cache_audit import adopt_existing_stage_1_6_cache_metadata
# display(adopt_existing_stage_1_6_cache_metadata(CONFIG))


cache_audit = stage_1_6_cache_audit(CONFIG)
cache_misses = cache_audit.filter(~pl.col("cache_hit"))

stage0_5_report = write_stage_report(
    "00_5",
    "Expensive Cache Audit",
    summary=[
        "Checks whether expensive Stage 1-6 artifacts can be reused safely.",
        "Artifacts with cache_hit=False will be rebuilt when their stage runs.",
    ],
    metrics={
        "audited_artifacts": cache_audit.height,
        "cache_ready_artifacts": cache_audit.filter(pl.col("cache_hit")).height,
        "rebuild_artifacts": cache_misses.height,
    },
    artifacts={"Artifact metadata manifest": CONFIG.outputs / ".artifact_metadata.json"},
    cfg=CONFIG,
)
display_stage_report(stage0_5_report)
cache_display_cols = ["stage", "artifact", "cache_hit", "exists", "reason", "path"]
display(cache_audit.select([c for c in cache_display_cols if c in cache_audit.columns]).head(20))

if cache_misses.height:
    print("Artifacts listed as cache_hit=False will be rebuilt if their stage is run.")
else:
    print("All expensive Stage 1-6 artifacts are cache-ready.")


<a id="stage-1"></a>

## Stage 1: Basket Construction

Each ticket is treated as a basket sentence and each product id is a token. Products are not repeated by quantity unless `baskets.repeat_product_by_quantity` is enabled in the config.

In [ ]:
from IPython.display import Image, display
import polars as pl

from src.basket_builder import (
    basket_summary,
    build_basket_sentences,
    build_basket_staple_diagnostics,
)
from src.stage_reports import display_stage_report, write_stage_report
from src.visualization import (
    plot_basket_staple_diagnostics,
    plot_basket_summary,
)

basket_path = build_basket_sentences(transactions=transactions, cfg=CONFIG)
stage1_basket_summary = basket_summary(basket_path)
stage1_figure = plot_basket_summary(basket_path, cfg=CONFIG)

stage1_diagnostics = build_basket_staple_diagnostics(transactions=transactions, cfg=CONFIG)
stage1_staple_figure = plot_basket_staple_diagnostics(
    stage1_diagnostics["product_diagnostics"],
    stage1_diagnostics["basket_exposure"],
    cfg=CONFIG,
)
stage1_common_products = (
    pl.read_csv(stage1_diagnostics["common_products_csv"]).height
    if stage1_diagnostics["common_products_csv"].exists()
    else 0
)
stage1_metrics = stage1_basket_summary.row(0, named=True)
stage1_metrics["common_product_candidates"] = stage1_common_products
stage1_report = write_stage_report(
    "01",
    "Basket Sentences and Common-Product Exposure",
    summary=[
        "Baskets are built from product identity and unidades only.",
        "Common-product exposure is audited so staples do not cloud downstream tribes.",
    ],
    metrics=stage1_metrics,
    figures={
        "Basket token summary": stage1_figure,
        "Common-product exposure": stage1_staple_figure,
    },
    artifacts={
        "Basket sentences": basket_path,
        "Common-product candidates CSV": stage1_diagnostics["common_products_csv"],
        "Product ubiquity diagnostics": stage1_diagnostics["product_diagnostics"],
    },
    cfg=CONFIG,
)
display_stage_report(stage1_report)
display(stage1_basket_summary)
display(Image(filename=str(stage1_figure)))
display(Image(filename=str(stage1_staple_figure)))
print(f"Stage 2 Item2Vec basket path: {basket_path}")


<a id="stage-2"></a>

## Stage 2: Item2Vec Product Embeddings

The Word2Vec model learns product proximity from basket co-occurrence. These product embeddings are the core signal used to represent customers.

The training function prints the active hyperparameters and one progress line per epoch. If a cached model already exists, it prints the cache details instead; pass `force=True` to retrain.

In [ ]:
from IPython.display import Image
import polars as pl

from src.item2vec import save_product_embeddings, train_item2vec
from src.stage_reports import display_stage_report, write_stage_report
from src.visualization import plot_product_embedding_diagnostics

item2vec_model = train_item2vec(basket_path, cfg=CONFIG, verbose=True)
product_embeddings_path = save_product_embeddings(item2vec_model, cfg=CONFIG)
stage2_figure = plot_product_embedding_diagnostics(product_embeddings_path, cfg=CONFIG)
stage2_schema = pl.read_parquet(product_embeddings_path, n_rows=1).columns
stage2_metrics = {
    "embedded_products": pl.scan_parquet(product_embeddings_path).select(pl.len()).collect()[0, 0],
    "embedding_dimensions": len([c for c in stage2_schema if c.startswith("emb_")]),
    "word2vec_window": CONFIG.get("word2vec.window"),
    "word2vec_min_count": CONFIG.get("word2vec.min_count"),
}
stage2_report = write_stage_report(
    "02",
    "Product Embedding Training",
    summary=[
        "Item2Vec is trained on Stage 1 product-token baskets.",
        "The training signal remains product identity plus unidades-derived basket tokens.",
    ],
    metrics=stage2_metrics,
    figures={"Product embedding diagnostics": stage2_figure},
    artifacts={"Product embeddings": product_embeddings_path},
    cfg=CONFIG,
)
display_stage_report(stage2_report)
display(Image(filename=str(stage2_figure)))


<a id="stage-3"></a>

## Stage 3: Product Embedding Validation

Before clustering customers, the nearest-neighbor report checks whether embeddings capture meaningful substitutes, complements, or shared basket missions.

In [ ]:
from IPython.display import Image
import polars as pl

from src.embedding_validation import product_embedding_guardrail_status, validate_product_embeddings
from src.stage_reports import display_stage_report, write_stage_report
from src.visualization import plot_embedding_validation_quality_extracts, plot_embedding_validation_summary

embedding_validation_csv, embedding_validation_detail_md = validate_product_embeddings(
    product_embeddings_path,
    transactions=transactions,
    cfg=CONFIG,
)
stage3_figure = plot_embedding_validation_summary(embedding_validation_csv, cfg=CONFIG)
stage3_extract_figures = {}
if CONFIG.get("embedding_validation.write_extract_figures", False):
    stage3_extract_figures = plot_embedding_validation_quality_extracts(embedding_validation_csv, cfg=CONFIG)
stage3_guardrail_status = product_embedding_guardrail_status(embedding_validation_csv, cfg=CONFIG)
stage3_figures = {"Embedding validation summary": stage3_figure}
stage3_figures.update({f"Quality extract: {name}": path for name, path in stage3_extract_figures.items()})
stage3_issues = stage3_guardrail_status["issues"] or ["None"]
stage3_report = write_stage_report(
    "03",
    "Product Embedding Validation",
    summary=[
        f"Guardrail status: {stage3_guardrail_status['status']}",
        stage3_guardrail_status["summary"],
        "Issues: " + "; ".join(stage3_issues[:4]),
    ],
    metrics={
        "validated_products": pl.read_csv(embedding_validation_csv).height,
        "guardrail_issue_count": 0 if stage3_guardrail_status["issues"] is None else len(stage3_guardrail_status["issues"]),
    },
    figures=stage3_figures,
    artifacts={
        "Validation CSV": embedding_validation_csv,
        "Hubness CSV": stage3_guardrail_status["hubness_csv"],
    },
    cfg=CONFIG,
)
display_stage_report(stage3_report)
display(Image(filename=str(stage3_figure)))
for figure_path in stage3_extract_figures.values():
    display(Image(filename=str(figure_path)))
stage3_guardrail_status


<a id="stage-4"></a>

## Stage 4: Customer Embeddings

Customer vectors are weighted means of product embeddings. The official weighting uses product quantities (`unidades`) only, with the configured `customer_embeddings.quantity_transform` applied before aggregation, so clustering is driven by products and purchase intensity rather than spend. IDF-downweighted variants are tested only in the experiment sandbox before any promotion to YAML.

In [ ]:
from IPython.display import Image, display
import polars as pl

from src.customer_embeddings import build_customer_embeddings
from src.stage_reports import display_stage_report, write_stage_report
from src.visualization import plot_customer_embedding_diagnostics

customer_embeddings_path = build_customer_embeddings(
    product_embeddings_path,
    transactions=transactions,
    normalize_vectors=CONFIG.get("customer_embeddings.normalize_vectors", False),
    cfg=CONFIG,
)
stage4_figure = plot_customer_embedding_diagnostics(customer_embeddings_path, cfg=CONFIG)

stage4_diag_cfg = CONFIG.get("customer_embeddings.diagnostics", {}) or {}
stage4_weight_summary_path = (
    CONFIG.artifacts
    / str(stage4_diag_cfg.get("output_dir", "stage4"))
    / f"{stage4_diag_cfg.get('output_prefix', 'customer_embedding')}_weight_diagnostics.csv"
)
stage4_display_cols = [
    "weight_strategy",
    "quantity_transform",
    "line_coverage_pct",
    "unit_coverage_pct",
    "product_coverage_pct",
    "customer_coverage_pct",
    "zero_embedded_customer_pct",
    "top_product_weight_share_pct",
    "top_10_product_weight_share_pct",
    "common_product_weight_share_pct",
    "mean_customer_top_product_weight_share_pct",
    "mean_customer_common_product_weight_share_pct",
    "coverage_gate_status",
    "dominance_gate_status",
]
stage4_weight_summary = None
stage4_report_metrics = {
    "embedded_customers": pl.scan_parquet(customer_embeddings_path).select(pl.len()).collect()[0, 0]
}
if stage4_weight_summary_path.exists():
    stage4_weight_summary = pl.read_csv(stage4_weight_summary_path)
    available_stage4_cols = [c for c in stage4_display_cols if c in stage4_weight_summary.columns]
    if available_stage4_cols:
        stage4_report_metrics.update(stage4_weight_summary.select(available_stage4_cols).row(0, named=True))
stage4_report = write_stage_report(
    "04",
    "Customer Embeddings Coverage and Dominance Gates",
    summary=[
        "Customer vectors are weighted aggregations of purchased-product embeddings.",
        "Coverage gates check how much transaction signal survives product embedding coverage.",
        "Dominance gates check whether common products overwhelm customer vectors.",
    ],
    metrics=stage4_report_metrics,
    figures={"Customer embedding diagnostics": stage4_figure},
    artifacts={
        "Customer embeddings": customer_embeddings_path,
        "Weight diagnostics": stage4_weight_summary_path if stage4_weight_summary_path.exists() else None,
    },
    cfg=CONFIG,
)
display_stage_report(stage4_report)
display(Image(filename=str(stage4_figure)))
if stage4_weight_summary is not None:
    display(stage4_weight_summary.select([c for c in stage4_display_cols if c in stage4_weight_summary.columns]))


In [ ]:
import numpy as np, polars as pl
ce = pl.read_parquet(customer_embeddings_path)
emb = [c for c in ce.columns if c.startswith("emb_")]
norms = np.linalg.norm(ce.select(emb).to_numpy(), axis=1)
print(f"norm min/mean/max: {norms.min():.3f} / {norms.mean():.3f} / {norms.max():.3f}")

<a id="stage-5"></a>

## Stage 5: Behavioral Features

Behavioral features are stored separately so the analysis can compare embeddings-only segmentation against embeddings plus behavior without allowing KPIs to silently dominate the product signal.

In [ ]:
from IPython.display import Image

from src.feature_engineering import build_behavioral_features, build_feature_set
from src.stage_reports import display_stage_report, write_stage_report
from src.visualization import plot_behavioral_feature_summary, plot_feature_set_summary

behavior_path = build_behavioral_features(transactions=transactions, cfg=CONFIG)
feature_set_a = build_feature_set(customer_embeddings_path, variant="embeddings_only", cfg=CONFIG)
feature_set_b = build_feature_set(
    customer_embeddings_path,
    behavior_path=behavior_path,
    variant="embeddings_behavior",
    cfg=CONFIG,
)
feature_sets = {
    "embeddings_only": feature_set_a,
    "embeddings_behavior": feature_set_b,
}
stage5_behavior_figure = plot_behavioral_feature_summary(behavior_path, cfg=CONFIG)
stage5_feature_set_figure = plot_feature_set_summary(feature_sets, cfg=CONFIG)
stage5_report = write_stage_report(
    "05",
    "Behavioral Features and Feature Sets",
    summary=[
        "Official tribe selection can use embeddings_only to keep clustering product-first.",
        "Behavioral features are prepared for comparison and post-clustering interpretation.",
    ],
    metrics={
        "feature_sets_built": len(feature_sets),
        "selection_feature_set": CONFIG.get("modeling.feature_set_for_selection", "embeddings_only"),
    },
    figures={
        "Behavioral feature summary": stage5_behavior_figure,
        "Feature set summary": stage5_feature_set_figure,
    },
    artifacts={"Behavioral features": behavior_path, **feature_sets},
    cfg=CONFIG,
)
display_stage_report(stage5_report)
display(Image(filename=str(stage5_behavior_figure)))
display(Image(filename=str(stage5_feature_set_figure)))


<a id="stage-6"></a>

## Stage 6: Candidate Model Comparison

The official pipeline compares algorithm families without forcing a target tribe count. Broader hyperparameter sweeps belong in `04_experiment_sandbox.ipynb`; promote only evidence-backed settings into YAML before running this notebook.

- `model_a_gmm`: Raw customer embeddings to Gaussian Mixture Model, searched across the configured `gmm.components_min` to `gmm.components_max` range.
- `model_b_umap_hdbscan`: UMAP customer manifold to HDBSCAN using the active YAML `umap` and `hdbscan` settings.
- `model_c_pca_kmeans`: PCA representation to MiniBatchKMeans, searched across the configured `kmeans.k_min` to `kmeans.k_max` range.

UMAP is treated as a candidate clustering representation, not as an automatic winner. Its value must be proven by metrics, stability, and product-lift interpretability.

For the promoted UMAP-HDBSCAN candidate, this stage creates the first two layers of the tribe flow. HDBSCAN identifies the organic core tribes; optional soft assignment then attaches nearby noise customers for operational coverage while retaining `assignment_source`, `assignment_confidence_score`, and `assignment_confidence_type`.

In [ ]:
import polars as pl
from IPython.display import Image, display

from src.model_selection import build_candidate_model_diagnostics, run_candidate_model_suite
from src.stage_reports import display_stage_report, write_stage_report
from src.visualization import plot_stage6_model_diagnostics

if "feature_sets" not in globals():
    feature_set_outputs = CONFIG.get("feature_sets.outputs", {})
    feature_sets = {
        name: CONFIG.outputs / "features" / filename
        for name, filename in feature_set_outputs.items()
    }
    missing_feature_sets = [path for path in feature_sets.values() if not path.exists()]
    if missing_feature_sets:
        raise FileNotFoundError(
            "Stage 6 needs the Stage 5 feature-set parquet files. "
            f"Missing: {missing_feature_sets}. Run Stage 5 first."
        )

selection_feature_set = CONFIG.get("modeling.feature_set_for_selection", "embeddings_only")
model_suite = run_candidate_model_suite(feature_sets[selection_feature_set], cfg=CONFIG)
stage6_diagnostics = build_candidate_model_diagnostics(model_suite, cfg=CONFIG)
stage6_figure = plot_stage6_model_diagnostics(stage6_diagnostics["parquet"], cfg=CONFIG)
stage6_table = pl.read_parquet(stage6_diagnostics["parquet"]).select([
    "stage6_rank",
    "model_name",
    "algorithm_name",
    "model_variant",
    "cluster_count",
    "coverage_adjusted_silhouette",
    "silhouette",
    "davies_bouldin",
    "noise_pct",
    "passes_quality_gate",
]).sort("stage6_rank")
stage6_best = stage6_table.row(0, named=True)
stage6_report = write_stage_report(
    "06",
    "Candidate Model Diagnostics",
    summary=[
        f"Selection feature set: {selection_feature_set}",
        f"Best candidate: {stage6_best['model_name']} / {stage6_best['model_variant']}",
    ],
    metrics=stage6_best,
    figures={"Model diagnostics": stage6_figure},
    artifacts={
        "Diagnostics parquet": stage6_diagnostics["parquet"],
        "Diagnostics summary CSV": stage6_diagnostics["summary_csv"],
    },
    cfg=CONFIG,
)
display_stage_report(stage6_report)
display(Image(filename=str(stage6_figure)))
display(stage6_table.head(12))


<a id="stage-7"></a>

## Stage 7: Cluster Validity and Stability

Stage 6 chooses the best candidate from model-quality metrics. Stage 7 is the guardrail before interpretation: it reviews cluster count, noise, balance, and a lightweight perturbation check that asks whether labels remain recoverable from the original customer-product feature geometry.

Use this stage to answer: does the Stage 6 winner still look defensible when we check robustness and label recoverability? The output is not a client story yet; it is the technical evidence that prevents us from over-interpreting a weak segmentation.

The notebook-facing summary is written to `outputs/<mode>/reports/stage_07.md`, with the winner PCA/UMAP maps displayed directly below. Detailed model-selection tables remain available as supporting artifacts.


In [ ]:
from IPython.display import Image, display
import polars as pl

from src.cluster_validation import build_cluster_validity_stability_report
from src.stage_reports import display_stage_report, write_stage_report
from src.visualization import build_2d_projection_figures

cluster_validity = build_cluster_validity_stability_report(
    model_suite,
    feature_sets[selection_feature_set],
    cfg=CONFIG,
)

validity_table = pl.read_csv(cluster_validity["summary_csv"]).select([
    "candidate_id",
    "cluster_count",
    "noise_pct",
    "cluster_size_cv",
    "jitter_ari_mean",
    "jitter_ari_std",
    "jitter_label_recovery_accuracy_mean",
    "validity_note",
])
stage6_winner_key = pl.read_parquet(stage6_diagnostics["parquet"]).sort("stage6_rank")[0, "candidate_id"]
stage7_winner_validity = validity_table.filter(pl.col("candidate_id") == stage6_winner_key)

stage7_projection_dir = CONFIG.figures
stage7_projection_figures = build_2d_projection_figures(
    feature_sets[selection_feature_set],
    model_suite["assignment_paths"][stage6_winner_key],
    output_dir=stage7_projection_dir,
    output_prefix="stage_07_winner_projection",
    stage_label="Stage 7 figures",
    cfg=CONFIG,
)
stage7_report = write_stage_report(
    "07",
    "Cluster Validity, Stability, and Projection Check",
    summary=[
        f"Stage 6 winner checked: {stage6_winner_key}",
        "Stability and 2D projections are evidence, not automatic winners.",
    ],
    metrics=stage7_winner_validity,
    figures={
        "PCA projection": stage7_projection_figures.get("pca"),
        "UMAP projection": stage7_projection_figures.get("umap"),
    },
    artifacts={
        "Validity/stability parquet": cluster_validity["parquet"],
        "Validity/stability summary CSV": cluster_validity["summary_csv"],
        "Projection figures root": stage7_projection_dir,
    },
    cfg=CONFIG,
)
display_stage_report(stage7_report)
display(validity_table.head(20))
display(stage7_winner_validity)
if "umap" in stage7_projection_figures:
    display(Image(filename=str(stage7_projection_figures["umap"])))
display(Image(filename=str(stage7_projection_figures["pca"])))


<a id="stage-8"></a>

## Stage 8: Tribe Profiling and Interpretability

Only the Stage 6 winner is profiled using product lift, sector lift, strategic product-theme lift, and data-driven product-term lift. Raw product popularity is not enough because staples tend to dominate all customers; lift tells us what is distinctive inside the selected tribe solution relative to the assigned population.

This stage intentionally keeps modeling and interpretation separate. Spend and behavior KPIs can support the business read, but the tribe signal remains product-first.

This stage produces the evidence used to explain what makes the selected tribe solution distinctive. The main dashboard compares each tribe with the rest of the assigned population through size, assignment provenance, lifted product signals, and behavior/KPI ratios. Non-winning candidate profiling belongs in the sandbox, not the official pipeline.

Primary output:

- tribe profile parquet with product, sector, theme, term, KPI, and assignment evidence


In [ ]:
import polars as pl
from IPython.display import Image, display

from src.profiling import (
    profile_evidence_metrics_table,
    profile_overview_table,
    profile_quality_summary,
    profile_subsegment_opportunity_table,
    profile_tribes,
)
from src.stage_reports import display_stage_report, write_stage_report
from src.visualization import plot_tribe_theme_lift_heatmap, plot_tribe_vs_population_evidence_dashboard

stage6_ranked = pl.read_parquet(stage6_diagnostics["parquet"]).sort("stage6_rank")
stage6_winner = stage6_ranked.row(0, named=True)
selected_key = stage6_winner["candidate_id"]
selected_assignment_path = model_suite["assignment_paths"][selected_key]
selected_profile_path = profile_tribes(
    selected_assignment_path,
    transactions=transactions,
    behavior_path=behavior_path,
    cfg=CONFIG,
)
profile_paths = {selected_key: selected_profile_path}
stage8_figures_dir = CONFIG.figures
stage8_figures_dir.mkdir(parents=True, exist_ok=True)

tribe_vs_population_dashboard = plot_tribe_vs_population_evidence_dashboard(
    selected_profile_path,
    output_path=stage8_figures_dir / f"stage_08_tribe_vs_population_evidence_dashboard_{CONFIG.mode}.png",
    cfg=CONFIG,
)
theme_lift_heatmap = plot_tribe_theme_lift_heatmap(
    selected_profile_path,
    output_path=stage8_figures_dir / f"stage_08_tribe_theme_lift_heatmap_{CONFIG.mode}.png",
    cfg=CONFIG,
)
stage8_model_table = pl.DataFrame([
    {
        "stage6_rank": stage6_winner.get("stage6_rank"),
        "candidate_id": selected_key,
        "cluster_count": stage6_winner.get("cluster_count"),
        "profile_path": str(selected_profile_path),
    }
])
stage8_quality = profile_quality_summary(selected_profile_path, cfg=CONFIG)
stage8_report = write_stage_report(
    "08",
    "Tribe Profiling Evidence",
    summary=[
        f"Profiled selected Stage 6 model: {selected_key}",
        "Profiles use spend/KPIs for interpretation after clustering, not as clustering signal.",
    ],
    metrics=stage8_quality,
    figures={
        "Tribe vs population dashboard": tribe_vs_population_dashboard,
        "Theme lift heatmap": theme_lift_heatmap,
    },
    artifacts={"Selected profile": selected_profile_path},
    cfg=CONFIG,
)
display_stage_report(stage8_report)
display(stage8_model_table)
display(pl.DataFrame([stage8_quality]))
display(Image(filename=str(tribe_vs_population_dashboard)))
display(Image(filename=str(theme_lift_heatmap)))

display(profile_overview_table(selected_profile_path, cfg=CONFIG).head(20))
display(profile_evidence_metrics_table(selected_profile_path, cfg=CONFIG).head(20))
display(profile_subsegment_opportunity_table(selected_profile_path, max_rows=20, cfg=CONFIG))

selected_profile_path


<a id="stage-9"></a>

## Stage 9: Final Model Selection and Shopping Mission Summary

The final section selects the recommended model, exports the clean assignment/profile files, and creates a small set of presentation tables and figures. The notebook-facing summary is written to `outputs/<mode>/reports/stage_09.md`; detailed tables remain available as supporting artifacts.

The presentation should stay simple: core tribes first, assignment confidence second, and a capped set of strongly backed shopping missions third.

The final outputs should be read as a progression: core tribe discovery, confidence-scored assignment coverage, then selected shopping missions. This avoids presenting exploratory overlays as if they were equally strong as the organic core tribes.

For client activation, the shopping mission layer flips the view: it starts from specific product-backed missions such as special diet, organic/bio, world cuisine, pets, baby/kids, meat/seafood, drinks, or wellness. The notebook only promotes the largest/strongest mission rows that pass the configured evidence filter, because the final set can change in production mode.

Important: tribe and mission labels are purchase-evidence summaries, not demographic, religious, household, or identity claims.


In [ ]:
import polars as pl
from IPython.display import Image, display

from src.exports import (
    build_model_comparison,
    collect_cached_presentation_figures,
    evidence_report_dir,
    export_final_assignments,
    export_final_profiles,
    merge_stage9_figure_paths,
    presentation_report_dir,
    selected_model,
    write_decision_log,
    write_stage9_presentation_pack,
)
from src.mission_microtribes import write_mission_microtribe_artifacts
from src.profiling import (
    write_clustering_atlas_artifacts,
    write_cluster_summary_artifacts,
)
from src.stage_reports import display_stage_report, write_stage_report
from src.visualization import (
    build_2d_projection_figures,
    plot_assignment_provenance,
    plot_cluster_sizes,
    plot_core_mission_lift_heatmap,
    plot_mission_microtribe_overview,
)

presentation_dir = presentation_report_dir(CONFIG)
evidence_dir = evidence_report_dir(CONFIG)
presentation_figures_dir = CONFIG.figures
presentation_dir.mkdir(parents=True, exist_ok=True)
evidence_dir.mkdir(parents=True, exist_ok=True)
presentation_figures_dir.mkdir(parents=True, exist_ok=True)

comparison_path = build_model_comparison(
    model_suite["candidate_results"],
    profile_paths=profile_paths,
    output_path=evidence_dir / f"model_comparison_{CONFIG.mode}.csv",
    cfg=CONFIG,
)
selected = selected_model(comparison_path)
selected_key = f"{selected['model_name']}::{selected['model_variant']}"
selected_assignment_path = model_suite["assignment_paths"][selected_key]
if selected_key not in profile_paths:
    raise RuntimeError(
        "Stage 9 selected a model that Stage 8 did not profile. "
        "Inspect the Stage 6 diagnostics/model comparison before exporting final tribes."
    )
selected_profile_path = profile_paths[selected_key]

assignment_export = export_final_assignments(
    selected_assignment_path,
    output_path=evidence_dir / f"customer_tribe_assignments_{CONFIG.mode}.parquet",
    cfg=CONFIG,
)
profile_export = export_final_profiles(
    selected_profile_path,
    output_path=evidence_dir / f"tribe_profiles_{CONFIG.mode}.csv",
    cfg=CONFIG,
)
decision_log_path = write_decision_log(
    comparison_path,
    selected_profile_path,
    output_path=evidence_dir / f"decision_log_{CONFIG.mode}.txt",
    cfg=CONFIG,
)
cluster_summary_paths = write_cluster_summary_artifacts(
    selected_profile_path,
    output_csv=presentation_dir / f"01_core_tribes_{CONFIG.mode}.csv",
    cfg=CONFIG,
)
mission_microtribe_paths = write_mission_microtribe_artifacts(
    assignment_export,
    transactions=transactions,
    core_tribe_summary_path=cluster_summary_paths["csv"],
    output_customer_parquet=evidence_dir / f"customer_mission_tags_{CONFIG.mode}.parquet",
    output_summary_csv=presentation_dir / f"02_shopping_missions_{CONFIG.mode}.csv",
    cfg=CONFIG,
)
figures = {
    "cluster_sizes": plot_cluster_sizes(
        selected_assignment_path,
        output_path=presentation_figures_dir / f"stage_09_core_tribe_sizes_{CONFIG.mode}.png",
        cfg=CONFIG,
    ),
    "assignment_provenance": plot_assignment_provenance(
        assignment_export,
        output_path=presentation_figures_dir / f"stage_09_assignment_provenance_{CONFIG.mode}.png",
        cfg=CONFIG,
    ),
    "projection": build_2d_projection_figures(
        feature_sets[selection_feature_set],
        selected_assignment_path,
        output_dir=presentation_figures_dir,
        output_prefix="stage_09_final_projection",
        cfg=CONFIG,
    ),
    "mission_overview": plot_mission_microtribe_overview(
        mission_microtribe_paths["summary_csv"],
        output_path=presentation_figures_dir / f"stage_09_shopping_mission_overview_{CONFIG.mode}.png",
        cfg=CONFIG,
    ),
    "core_mission_lift_heatmap": plot_core_mission_lift_heatmap(
        mission_microtribe_paths["parquet"],
        cluster_summary_paths["csv"],
        output_path=presentation_figures_dir / f"stage_09_core_tribe_by_shopping_mission_lift_{CONFIG.mode}.png",
        cfg=CONFIG,
    ),
}
figure_manifest = merge_stage9_figure_paths(
    collect_cached_presentation_figures(cfg=CONFIG),
    figures,
)
clustering_atlas_paths = write_clustering_atlas_artifacts(
    selected_profile_path,
    figure_paths=figure_manifest,
    output_csv=presentation_dir / f"03_clustering_atlas_{CONFIG.mode}.csv",
    output_html=presentation_dir / f"03_clustering_atlas_{CONFIG.mode}.html",
    write_markdown=False,
    cfg=CONFIG,
)
presentation_pack_paths = write_stage9_presentation_pack(
    selected=selected,
    core_summary_path=cluster_summary_paths["csv"],
    mission_summary_path=mission_microtribe_paths["summary_csv"],
    clustering_atlas_path=clustering_atlas_paths["html"],
    assignment_path=assignment_export,
    profile_path=profile_export,
    figure_paths=figure_manifest,
    evidence_paths={
        "model_comparison": comparison_path,
        "decision_log": decision_log_path,
        "profile_export": profile_export,
    },
    output_dir=presentation_dir,
    cfg=CONFIG,
)
stage9_report_figures = {
    "UMAP 2D customer map": figures["projection"].get("umap"),
    "PCA 2D customer map": figures["projection"].get("pca"),
    "Core tribe sizes": figures["cluster_sizes"],
    "Assignment provenance": figures["assignment_provenance"],
    "Shopping mission overview": figures["mission_overview"],
    "Core tribe by mission lift": figures["core_mission_lift_heatmap"],
}
stage9_report = write_stage_report(
    "09",
    "Final Tribe Exports and Presentation Pack",
    summary=[
        f"Selected model: {selected['model_name']} ({selected['model_variant']})",
        f"Selected tribe count: {selected['cluster_count']}",
        "Recommended flow: core tribes, assignment confidence, selected shopping missions, then evidence tables if challenged.",
    ],
    metrics={
        "core_tribe_rows": pl.read_csv(cluster_summary_paths["csv"]).height,
        "shopping_mission_rows": pl.read_csv(mission_microtribe_paths["summary_csv"]).height,
        "presentation_figures": len([path for path in stage9_report_figures.values() if path is not None]),
    },
    figures=stage9_report_figures,
    artifacts={
        "Core tribe table": cluster_summary_paths["csv"],
        "Selected shopping missions": mission_microtribe_paths["summary_csv"],
        "Clustering atlas": clustering_atlas_paths["html"],
        "Presentation pack": presentation_pack_paths["html"],
        "Artifact manifest": presentation_pack_paths["manifest"],
        "Decision log": decision_log_path,
        "Operational assignment export": assignment_export,
    },
    cfg=CONFIG,
)
display_stage_report(stage9_report)
if "umap" in figures["projection"]:
    display(Image(filename=str(figures["projection"]["umap"])))
if "pca" in figures["projection"]:
    display(Image(filename=str(figures["projection"]["pca"])))
display(Image(filename=str(figures["cluster_sizes"])))
display(Image(filename=str(figures["assignment_provenance"])))
display(Image(filename=str(figures["mission_overview"])))
display(Image(filename=str(figures["core_mission_lift_heatmap"])))

display(pl.read_csv(cluster_summary_paths["csv"]).head(20))
display(pl.read_csv(mission_microtribe_paths["summary_csv"]).head(20))


<a id="decision-log"></a>

## Decision Log: Final Tribe Model Selection

This final section is generated after model evaluation and profiling. It records which model was selected, how many tribes were selected, whether that result agrees with the 10-15 tribe client hypothesis, why the selected model won, why alternatives were rejected, what evidence supports the recommendation, remaining limitations, and next production rollout steps.

In [ ]:
from src.stage_reports import display_stage_report

if "stage9_report" in globals():
    display_stage_report(stage9_report, max_lines=80)
else:
    display_stage_report(decision_log_path, max_lines=60)


<a id="stage-10"></a>

## Stage 10: Additional Business Lens

This section turns the modeling result back toward the original Carrefour business problem: the company needs customer groups that are specific enough to target with relevant campaigns, offers, retention actions, and category growth plays.

The clustering itself remains product-first and quantity-based. Spend, promo behavior, and KPIs are used only after clustering to size and prioritize opportunities.

The business lens reads the final solution as an activation stack:

1. Core tribes define the strategic customer groups.
2. Assignment confidence tells Carrefour how safely to target each customer.
3. Shopping missions create product-backed campaign audiences that can overlap.
4. Financial sizing translates audiences into testable opportunity ranges, not guaranteed ROI.

The resulting tables should support campaign briefs such as: who to target, why they are targetable, what product evidence supports the message, which KPI to measure, and how to size the commercial upside with a control group.

In [ ]:
import polars as pl
from IPython.display import display

from src.business_lens import write_business_lens_artifacts
from src.stage_reports import display_stage_report, write_stage_report

business_lens_paths = write_business_lens_artifacts(
    core_summary_path=cluster_summary_paths["csv"],
    mission_summary_path=mission_microtribe_paths["summary_csv"],
    profile_path=profile_export,
    output_dir=presentation_dir,
    write_markdown=False,
    cfg=CONFIG,
)
stage10_report = write_stage_report(
    "10",
    "Business Lens",
    summary=[
        "Transforms the selected product-first tribes into business-facing interpretation.",
        "This stage is post-clustering interpretation; it does not alter tribe discovery.",
    ],
    metrics={"business_lens_artifacts": len(business_lens_paths)},
    artifacts=business_lens_paths,
    cfg=CONFIG,
)
display_stage_report(stage10_report)
display(
    pl.DataFrame(
        [
            {"artifact": key, "path": str(path)}
            for key, path in business_lens_paths.items()
        ]
    )
)
